# Cool Roof Priority Index — Mussafah, Abu Dhabi

**Arab Youth Space Hackathon 2026 · 813 Challenge**
**Theme:** Urban Expansion, Land Use Change & Heat Risk · **Track:** Hyperspectral
**Team:** T0050 · United Arab Emirates

---

### What this notebook produces

A ranked worklist of 3,603 individual rooftops, with the cooling each would gain from
a cool-roof retrofit **measured from Abu Dhabi's own data** rather than imported from a
study elsewhere.

| § | Step | Output |
|---|------|--------|
| 3 | Landsat C2 L2 surface temperature + optical, both epochs | LST maps, ΔLST |
| 4 | Built-up classification: index baseline → random forest with texture | validated land cover |
| 5 | Urban expansion and its thermal cost | transition table, significance tests |
| 6 | Sentinel-2 broadband albedo | albedo map |
| 7 | **Calibrate the lever** — regress LST on albedo | **°C per unit albedo + bootstrap CI** |
| 8 | Rank the roofs | ranked CSV, GeoJSON, three ranking modes |
| 9 | Physical impact and exports | avoided solar absorption, summary.json |

Hyperspectral material identification runs separately in
`04_emit_hyperspectral.ipynb` (NASA EMIT over the same AOI).

### Three decisions worth defending

**Landsat alone carries the change analysis.** Both epochs, thermal and optical, from
one instrument family. Mixing Landsat 2014 with Sentinel-2 2025 would fold a
cross-sensor calibration difference into what we report as real-world change.

**Level-2, not Level-1.** USGS ships an atmospherically corrected Surface Temperature
product. Using it removes a class of hand-made assumptions and gives a citable chain.

**Validation is spatial, never random.** Neighbouring pixels are near-duplicates; a
random train/test split leaks them across the boundary and reports a meaningless score.

**Runtime ≈ 35–45 min** on a free Colab CPU instance. No GPU.


## 1 · Environment

In [ ]:
%pip install -q \
    pystac-client==0.8.6 planetary-computer==1.0.0 stackstac==0.5.1 \
    rioxarray==0.17.0 rasterstats==0.20.0 scikit-learn==1.5.2 \
    geopandas==1.0.1 requests==2.32.4 scipy

In [ ]:
import os, sys, json, time, warnings, subprocess
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd
import matplotlib.pyplot as plt
import pystac_client, planetary_computer, stackstac, rioxarray, rasterio
from shapely.geometry import box, Polygon
from scipy.ndimage import uniform_filter
from scipy import stats as sps
from rasterstats import zonal_stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error
try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110

# stackstac + NumPy 2: fill_value must be a NUMPY scalar of the output dtype.
# Under NEP 50 np.can_cast(float, float32) is False, so a plain np.nan is rejected.
NODATA = np.float32("nan")

# Output folders, wherever this runs
if os.path.isdir("../results") and os.path.isdir("../data"):
    BASE = ".."
elif os.path.isdir("/content"):
    BASE = "/content/coolroof_outputs"
else:
    BASE = "."
RESULTS, SAMPLE = f"{BASE}/results", f"{BASE}/data/sample_input"
for d in (RESULTS, SAMPLE): os.makedirs(d, exist_ok=True)
print("outputs ->", os.path.abspath(RESULTS))

## 2 · Configuration

Every parameter the analysis depends on is in this one cell.

In [ ]:
AOI_NAME   = "Mussafah, Abu Dhabi"
AOI_BBOX   = [54.43, 24.29, 54.58, 24.41]     # W, S, E, N
FOCUS_BBOX = [54.48, 24.33, 54.55, 24.38]     # smaller core for the per-roof step

EPOCH_A, EPOCH_A_NAME = "2014-05-01/2016-09-30", "2014-2016"
EPOCH_B, EPOCH_B_NAME = "2024-05-01/2026-09-30", "2024-2026"
S2_WINDOW             = "2025-05-01/2026-09-30"     # albedo only, present day

MAX_CLOUD, MAX_SCENES_LS, MAX_SCENES_S2 = 15, 40, 20
EPSG, RES_LST, RES_S2 = 32640, 30, 20          # UTM 40N — correct zone for Abu Dhabi
TARGET_ALBEDO = 0.60                            # what a cool-roof coating reaches
WC_BUILTUP = 50                                 # ESA WorldCover built-up class

# Summer daytime-mean global horizontal irradiance, Abu Dhabi.
# TEAM: verify against Global Solar Atlas and cite in README section 5.
S_SUMMER_MEAN, S_NOON_PEAK = 300.0, 950.0       # W/m2

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace)
print(f"AOI    : {AOI_NAME} {AOI_BBOX}")
print(f"epochs : {EPOCH_A_NAME} vs {EPOCH_B_NAME}")

## 3 · Landsat Collection 2 Level-2 — temperature and optical, both epochs

One request per epoch pulls the thermal band and the optical bands together, so the
heat map and the land-cover map come from exactly the same scenes.

- Surface temperature: `lwir11`, `K = DN × 0.00341802 + 149.0`
- Surface reflectance: `DN × 0.0000275 − 0.2`
- Cloud, cirrus, shadow and dilated-cloud masking from `qa_pixel` **before** compositing

In [ ]:
LS_OPTICAL = ["green", "red", "nir08", "swir16"]
LS_ASSETS  = ["lwir11"] + LS_OPTICAL + ["qa_pixel"]
QA_BITS    = [0, 1, 2, 3, 4]      # fill, dilated cloud, cirrus, cloud, shadow

def landsat_epoch(bbox, window, label):
    items = list(catalog.search(
        collections=["landsat-c2-l2"], bbox=bbox, datetime=window,
        query={"eo:cloud_cover": {"lt": MAX_CLOUD},
               "platform": {"in": ["landsat-8", "landsat-9"]}}).items())
    if not items:
        raise RuntimeError(f"No Landsat scenes for {label}")
    items = sorted(items, key=lambda i: i.properties.get("eo:cloud_cover", 100))[:MAX_SCENES_LS]
    dates = sorted(i.properties["datetime"][:10] for i in items)
    print(f"{label}: {len(items)} cleanest scenes, {dates[0]} -> {dates[-1]} "
          f"(cloud {items[0].properties['eo:cloud_cover']:.1f}-"
          f"{items[-1].properties['eo:cloud_cover']:.1f}%)")

    stack = stackstac.stack(items, assets=LS_ASSETS, resolution=RES_LST,
                            bounds_latlon=bbox, epsg=EPSG, rescale=False,
                            dtype="float32", fill_value=NODATA, chunksize=1024)
    # fillna(1) sets the fill bit where the stack has no data, so missing pixels
    # are masked rather than reading as "no flags set".
    qa = stack.sel(band="qa_pixel").fillna(1).astype("uint16")
    bad = sum(((qa >> b) & 1) for b in QA_BITS)

    st = stack.sel(band="lwir11")
    lst = ((st.where(st > 0)*0.00341802 + 149.0) - 273.15).where(bad == 0)
    lst = lst.median(dim="time", skipna=True).compute(); lst.name = f"LST_{label}"

    sr = stack.sel(band=LS_OPTICAL)
    sr = ((sr*0.0000275 - 0.2).where(bad == 0)).median(dim="time", skipna=True).compute()
    print(f"   LST {np.nanmin(lst.values):.1f} to {np.nanmax(lst.values):.1f} C, "
          f"{int(np.isfinite(lst.values).sum()):,} valid px")
    return lst, sr, items

lst_a, sr_a, items_a = landsat_epoch(AOI_BBOX, EPOCH_A, EPOCH_A_NAME)
lst_b, sr_b, items_b = landsat_epoch(AOI_BBOX, EPOCH_B, EPOCH_B_NAME)
d_lst = (lst_b - lst_a).rename("delta_LST")
px_km2 = RES_LST**2 / 1e6

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, arr, t in zip(axes[:2], [lst_a, lst_b],
                      [f"Summer LST {EPOCH_A_NAME}", f"Summer LST {EPOCH_B_NAME}"]):
    im = ax.imshow(arr.values, cmap="inferno", vmin=35, vmax=60)
    ax.set_title(t); ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.046, label="C")
lim = float(np.nanpercentile(np.abs(d_lst.values), 98))
im = axes[2].imshow(d_lst.values, cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[2].set_title(f"Change {EPOCH_A_NAME} to {EPOCH_B_NAME}"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046, label="C")
plt.suptitle(f"Summer median land surface temperature — {AOI_NAME}",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS}/01_lst_change.png", bbox_inches="tight")
plt.show()
print(f"Mean LST {EPOCH_A_NAME}: {np.nanmean(lst_a.values):.2f} C")
print(f"Mean LST {EPOCH_B_NAME}: {np.nanmean(lst_b.values):.2f} C")

## 4 · Built-up classification — a documented escalation

We begin with the standard index threshold as the simple explainable baseline the
programme guidance asks for. **It fails here**, and we establish that rather than
assuming it: `NDBI = (SWIR − NIR)/(SWIR + NIR)`, and in hyper-arid terrain **bare
sabkha is SWIR-bright exactly like concrete**, so no threshold separates them.

Having documented the failure we escalate to a **random forest on 14 features including
multi-scale texture**. Industrial fabric is spatially rough at 30 m; sabkha is smooth.
That is the signal the spectral index cannot carry.

In [ ]:
eps = 1e-6
def indices(sr):
    g, r = sr.sel(band="green").values, sr.sel(band="red").values
    n, s = sr.sel(band="nir08").values, sr.sel(band="swir16").values
    return dict(ndvi=(n-r)/(n+r+eps), ndbi=(s-n)/(s+n+eps), mndwi=(g-s)/(g+s+eps))

def local_std(a, size=5):
    """Local standard deviation — a cheap, interpretable roughness measure."""
    a = np.where(np.isfinite(a), a, np.nanmean(a[np.isfinite(a)]))
    m  = uniform_filter(a.astype("float32"), size)
    m2 = uniform_filter((a*a).astype("float32"), size)
    return np.sqrt(np.clip(m2 - m*m, 0, None))

def features(sr, ix):
    g = sr.sel(band="green").values; r = sr.sel(band="red").values
    n = sr.sel(band="nir08").values; s = sr.sel(band="swir16").values
    bright = (g + r + n + s)/4.0
    return {"green": g, "red": r, "nir": n, "swir": s, "brightness": bright,
            "ndvi": ix["ndvi"], "ndbi": ix["ndbi"], "mndwi": ix["mndwi"],
            "bui": ix["ndbi"]-ix["ndvi"], "swir_nir_ratio": s/(n+eps),
            "tex_swir_5": local_std(s, 5), "tex_swir_9": local_std(s, 9),
            "tex_bright_5": local_std(bright, 5), "tex_ndbi_5": local_std(ix["ndbi"], 5)}

ix_a, ix_b = indices(sr_a), indices(sr_b)
F_a, F_b = features(sr_a, ix_a), features(sr_b, ix_b)
FEAT = list(F_b.keys())
Xb = np.stack([F_b[k] for k in FEAT], -1)
Xa = np.stack([F_a[k] for k in FEAT], -1)
ny, nx, nf = Xb.shape
print(f"{nf} features over {ny} x {nx} px")

In [ ]:
# ---- Reference: ESA WorldCover 2021 ----------------------------------------
wc_items = list(catalog.search(collections=["esa-worldcover"], bbox=AOI_BBOX).items())
wc_2021 = [i for i in wc_items if "2021" in i.id] or wc_items
print("reference:", [i.id for i in wc_2021])
wc = stackstac.stack(wc_2021, assets=["map"], resolution=RES_LST,
                     bounds_latlon=AOI_BBOX, epsg=EPSG, rescale=False,
                     dtype="float32", fill_value=NODATA).isel(band=0)
wc = (wc.max(dim="time") if "time" in wc.dims else wc).compute()
wc_grid = wc.interp_like(lst_b, method="nearest").values

y_ref = (wc_grid == WC_BUILTUP)
ok = np.isfinite(wc_grid) & np.isfinite(Xb).all(-1) & np.isfinite(Xa).all(-1)
print(f"usable px {ok.sum():,}   reference built-up share {y_ref[ok].mean()*100:.1f}%")

# ---- Spatial 2 km block split ----------------------------------------------
rows, cols = np.mgrid[0:ny, 0:nx]
BLK = int(2000/RES_LST)
blk = (rows//BLK)*10_000 + (cols//BLK)
ub = np.unique(blk); np.random.shuffle(ub)
is_tr = np.isin(blk, list(ub[:int(len(ub)*0.6)]))
tr, te = ok & is_tr, ok & ~is_tr
print(f"train {tr.sum():,} px | test {te.sum():,} px (disjoint 2 km blocks)")

Xf, yf = Xb.reshape(-1, nf), y_ref.ravel()
tr_idx = np.where(tr.ravel())[0]
if len(tr_idx) > 120_000:
    tr_idx = np.random.choice(tr_idx, 120_000, replace=False)
clf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features="sqrt",
                             class_weight="balanced_subsample", n_jobs=-1, random_state=42)
print("training random forest...")
clf.fit(Xf[tr_idx], yf[tr_idx])

In [ ]:
def metrics(pred, ref):
    TP=int((pred&ref).sum()); FP=int((pred&~ref).sum())
    FN=int((~pred&ref).sum()); TN=int((~pred&~ref).sum())
    p=TP/max(TP+FP,1); r=TP/max(TP+FN,1)
    return dict(TP=TP,FP=FP,FN=FN,TN=TN, accuracy=(TP+TN)/max(TP+TN+FP+FN,1),
                precision=p, recall=r, f1=2*p*r/max(p+r,1e-9), iou=TP/max(TP+FP+FN,1))

te_idx = np.where(te.ravel())[0]
pred_te = clf.predict(Xf[te_idx]).astype(bool)
ref_te  = yf[te_idx].astype(bool)
m_rf   = metrics(pred_te, ref_te)
m_ndbi = metrics(Xf[te_idx, FEAT.index("ndbi")] > 0.0, ref_te)

print("\n" + "="*72)
print("BUILT-UP CLASSIFICATION — held-out spatial blocks, never seen in training")
print("="*72)
print(f"{'metric':12s} {'NDBI > 0':>14s} {'random forest':>16s}")
for k in ["accuracy","precision","recall","f1","iou"]:
    print(f"{k:12s} {m_ndbi[k]:14.3f} {m_rf[k]:16.3f}")
print(f"{'FP':12s} {m_ndbi['FP']:14,} {m_rf['FP']:16,}")
print(f"\npredicted built-up share: baseline "
      f"{(Xf[te_idx, FEAT.index('ndbi')]>0).mean()*100:.1f}% | "
      f"forest {pred_te.mean()*100:.1f}% | reference {ref_te.mean()*100:.1f}%")
print("="*72)

imp = sorted(zip(FEAT, clf.feature_importances_), key=lambda t: -t[1])
print("\nFeature importance (top 8) — note where texture lands:")
for k, v in imp[:8]: print(f"   {k:16s} {v:.3f}")

validation_builtup = dict(
    method="random forest, 200 trees, 14 features incl. multi-scale texture",
    reference="ESA WorldCover 2021 v200, class 50",
    validation="2 km spatial blocks, 60/40 split, metrics on held-out blocks only",
    n_test=int(te.sum()),
    **{k:(round(float(v),4) if isinstance(v,float) else v) for k,v in m_rf.items()},
    baseline_ndbi_gt_0={k:(round(float(v),4) if isinstance(v,float) else v)
                        for k,v in m_ndbi.items()},
    top_features=[{"feature":k,"importance":round(float(v),4)} for k,v in imp[:8]],
    note=("Index thresholds were tried first and failed: calibrating the NDBI threshold "
          "on held-out data left precision at 0.40 at every threshold, because bare "
          "sabkha is SWIR-bright exactly like concrete. Texture separates them."))

In [ ]:
# ---- Apply the SAME model to both epochs ------------------------------------
print("predicting both epochs with one model...")
okf = ok.ravel()
bu_b = np.zeros(ny*nx, bool); bu_b[okf] = clf.predict(Xb.reshape(-1,nf)[okf]).astype(bool)
bu_a = np.zeros(ny*nx, bool); bu_a[okf] = clf.predict(Xa.reshape(-1,nf)[okf]).astype(bool)
bu_b, bu_a = bu_b.reshape(ny,nx), bu_a.reshape(ny,nx)

def compose(bu, ix):
    lc = np.full(bu.shape, 3, np.uint8)      # 3 bare / desert
    lc[bu] = 2                               # 2 built-up
    lc[ix["ndvi"] > 0.20] = 1                # 1 vegetation
    lc[ix["mndwi"] > 0.10] = 0               # 0 water
    return lc

lc_a, lc_b = compose(bu_a, ix_a), compose(bu_b, ix_b)
for lbl, lc in [(EPOCH_A_NAME, lc_a), (EPOCH_B_NAME, lc_b)]:
    sh = [(lc==i)[ok].mean()*100 for i in range(4)]
    print(f"  {lbl}: water {sh[0]:4.1f}%  veg {sh[1]:4.1f}%  "
          f"built {sh[2]:5.1f}%  desert {sh[3]:5.1f}%")

## 5 · Urban expansion and its thermal cost

We compare the temperature change of land converted from desert to built-up against
land that stayed desert. Both experienced the same regional climate, so the difference
isolates the urbanisation signal from the background warming trend.

In [ ]:
new_builtup    = (lc_a==3) & (lc_b==2) & ok
stable_builtup = (lc_a==2) & (lc_b==2) & ok
stable_desert  = (lc_a==3) & (lc_b==3) & ok
dl = d_lst.values

rows_ = []
for name, m in [("New built-up (was desert)", new_builtup),
                ("Stable built-up", stable_builtup), ("Stable desert", stable_desert)]:
    mm = m & np.isfinite(dl)
    rows_.append(dict(category=name, n_pixels=int(mm.sum()),
        area_km2=round(float(mm.sum()*px_km2),2),
        LST_before=round(float(np.nanmean(lst_a.values[mm])),2),
        LST_after=round(float(np.nanmean(lst_b.values[mm])),2),
        delta_LST=round(float(np.nanmean(dl[mm])),2),
        delta_LST_sd=round(float(np.nanstd(dl[mm])),2)))
transition = pd.DataFrame(rows_)
transition.to_csv(f"{RESULTS}/transition_summary.csv", index=False)
display(transition)

EXPANSION_KM2 = round(float(new_builtup.sum()*px_km2), 2)
groups = {"New built-up (was desert)": new_builtup & np.isfinite(dl),
          "Stable built-up": stable_builtup & np.isfinite(dl),
          "Stable desert": stable_desert & np.isfinite(dl)}

sig_rows = []
for A, B in [("New built-up (was desert)","Stable desert"),
             ("Stable built-up","Stable desert"),
             ("Stable built-up","New built-up (was desert)")]:
    a_, b_ = dl[groups[A]], dl[groups[B]]
    t = sps.ttest_ind(a_, b_, equal_var=False)
    d_eff = (a_.mean()-b_.mean())/np.sqrt((a_.var()+b_.var())/2)
    sig_rows.append(dict(contrast=f"{A} vs {B}", diff_c=round(float(a_.mean()-b_.mean()),3),
        t=round(float(t.statistic),1), p=f"{t.pvalue:.2e}", cohens_d=round(float(d_eff),3),
        verdict="SIGNIFICANT" if t.pvalue < 0.01 else "not significant"))
sig = pd.DataFrame(sig_rows); sig.to_csv(f"{RESULTS}/significance_tests.csv", index=False)

HEADLINE = sig_rows[0]["diff_c"]
lstB = lst_b.values
t_static = sps.ttest_ind(lstB[groups["Stable built-up"]], lstB[groups["Stable desert"]],
                         equal_var=False)
uhi_static = float(np.nanmean(lstB[groups["Stable built-up"]]) -
                   np.nanmean(lstB[groups["Stable desert"]]))

print("\n" + "="*78); print("SIGNIFICANCE — Welch t-tests on decadal LST change")
print("="*78)
display(sig)
print(f"Desert urbanised: {EXPANSION_KM2} km2 ({EXPANSION_KM2*100:,.0f} ha)")
print(f"Static surface UHI, {EPOCH_B_NAME}: built-up is {uhi_static:+.2f} C hotter than "
      f"desert (t={t_static.statistic:.1f}, p={t_static.pvalue:.2e})")

### Why conversion shows no extra warming — the diagnostic

The desert→built-up contrast is **not significant**. The reason is in the starting
temperatures: land that would later be built on was already much hotter than desert
that stayed desert, *before any construction*.

In [ ]:
def brightness(sr):
    return np.mean([sr.sel(band=b).values for b in LS_OPTICAL], axis=0)
bri_a, bri_b = brightness(sr_a), brightness(sr_b)

print("="*86)
print("What did the 'newly built' land look like BEFORE construction?")
print("="*86)
print(f"{'group':28s} {'n':>8s} {'bright A':>9s} {'NDBI A':>8s} {'LST A':>7s}")
diag = {}
for n, m in groups.items():
    d = dict(n=int(m.sum()), bright_a=float(np.nanmean(bri_a[m])),
             ndbi_a=float(np.nanmean(ix_a["ndbi"][m])),
             lst_a=float(np.nanmean(lst_a.values[m])))
    diag[n] = d
    print(f"{n:28s} {d['n']:8,} {d['bright_a']:9.3f} {d['ndbi_a']:8.3f} {d['lst_a']:7.2f}")

nb_, sd_ = diag["New built-up (was desert)"], diag["Stable desert"]
dT_pre = nb_["lst_a"] - sd_["lst_a"]
print(f"\nBefore construction, land later built on was {dT_pre:+.2f} C hotter than")
print(f"desert that stayed desert, with NDBI {nb_['ndbi_a']-sd_['ndbi_a']:+.3f} relative")
print("to it — spectrally almost identical.")
print("\nTwo explanations fit and we cannot separate them:")
print("  1. SITING — development happens next to existing urban fabric, so the land")
print("     was already inside the city's thermal footprint. A spatial confound.")
print("  2. SITE PREPARATION — clearing and compaction strip the salt crust and")
print("     shallow moisture of natural sabkha before any structure exists.")
print("Both are stated in the README. We claim neither.")

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))
cmap4 = ListedColormap(["#2196F3","#4CAF50","#9e9e9e","#f0e0c0"])
for ax, lc, t in zip(axes[:2], [lc_a, lc_b],
                     [f"Land cover {EPOCH_A_NAME}", f"Land cover {EPOCH_B_NAME}"]):
    ax.imshow(lc, cmap=cmap4, vmin=0, vmax=3); ax.axis("off"); ax.set_title(t)
axes[1].legend(handles=[Patch(color=c,label=l) for c,l in zip(cmap4.colors,
    ["Water","Vegetation","Built-up","Bare desert"])],
    loc="lower right", fontsize=8, framealpha=0.9)
axes[2].boxplot([dl[groups[n]] for n in groups],
                labels=["Newly\nbuilt-up","Stable\nbuilt-up","Stable\ndesert"],
                showfliers=False)
axes[2].axhline(0, color="k", lw=0.8, ls="--")
axes[2].set_ylabel("Decadal change in summer LST (C)")
axes[2].set_title(f"Change by transition\nconversion vs desert: p={sig_rows[0]['p']}")
axes[2].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(f"{RESULTS}/03_expansion_heat.png", bbox_inches="tight")
plt.show()

## 6 · Broadband albedo from Sentinel-2

Albedo is the physical property a cool roof changes, so it must be measured. Sentinel-2
is used here only — present day, no cross-epoch comparison — where its 20 m resolution
resolves individual large roofs.

```
α = 0.2266·B2 + 0.1236·B3 + 0.1573·B4 + 0.3417·B8 + 0.1170·B11 + 0.0338·B12
```

Weights sum to 1.0. Source: **Bonafoni & Sekertekin (2020)**, IEEE GRSL.

In [ ]:
S2_BANDS = ["B02","B03","B04","B08","B11","B12"]
ALB_W = dict(B02=0.2266, B03=0.1236, B04=0.1573, B08=0.3417, B11=0.1170, B12=0.0338)
SCL_BAD = [0,1,3,8,9,10,11]

items_s2 = list(catalog.search(collections=["sentinel-2-l2a"], bbox=AOI_BBOX,
    datetime=S2_WINDOW, query={"eo:cloud_cover":{"lt":MAX_CLOUD}}).items())
items_s2 = sorted(items_s2, key=lambda i: i.properties.get("eo:cloud_cover",100))[:MAX_SCENES_S2]
print(f"Sentinel-2: {len(items_s2)} cleanest scenes from {S2_WINDOW}")

s2 = stackstac.stack(items_s2, assets=S2_BANDS+["SCL"], resolution=RES_S2,
                     bounds_latlon=AOI_BBOX, epsg=EPSG, rescale=False,
                     dtype="float32", fill_value=NODATA, chunksize=1024)
good = ~s2.sel(band="SCL").isin(SCL_BAD)
refl = (s2.sel(band=S2_BANDS).where(good)/10000.0).median(dim="time", skipna=True).compute()
albedo = sum(w*refl.sel(band=n) for n,w in ALB_W.items()).rename("albedo").rio.write_crs(EPSG)
alb = albedo.values
print(f"albedo p1 {np.nanpercentile(alb,1):.3f}  median {np.nanmedian(alb):.3f}  "
      f"p99 {np.nanpercentile(alb,99):.3f}")

plt.figure(figsize=(8,6.5))
plt.imshow(alb, cmap="gray", vmin=0.05, vmax=0.55)
plt.colorbar(label="Broadband albedo", fraction=0.046)
plt.title(f"Surface albedo — {AOI_NAME}"); plt.axis("off")
plt.savefig(f"{RESULTS}/02_albedo.png", bbox_inches="tight"); plt.show()

## 7 · Calibrating the lever — how much cooling does albedo buy *here*?

The analytical core. Over pixels that **both** our classifier and ESA WorldCover
independently call built-up, with water and vegetation excluded, we regress summer LST
on albedo controlling for NDVI.

**Why the double mask matters.** Our first attempt used `NDBI > 0` alone and produced a
slope of **+11.55** — brighter surfaces appearing hotter, inverting the physics. The
cause was contamination: channel water (dark *and* cold) anchoring the low-albedo end,
and bright sabkha (bright *and* very hot) anchoring the high end. We keep that number in
the output because the audit trail is part of the evidence.

Uncertainty comes from a **300-sample spatial block bootstrap** — resampling whole 1 km
blocks, not pixels, because pixel resampling ignores spatial autocorrelation.

In [ ]:
def to20(a, method="linear"):
    return (xr.DataArray(np.asarray(a, dtype="float32"), dims=lst_b.dims,
                         coords=lst_b.coords).rio.write_crs(EPSG)
            .interp_like(albedo, method=method).values)

lst_20   = to20(lst_b.values)
ndvi_20  = to20(ix_b["ndvi"])
mndwi_20 = to20(ix_b["mndwi"])
wc_20    = to20(wc_grid, "nearest")
bu20     = to20(bu_b.astype("float32"), "nearest") > 0.5

finite = (np.isfinite(lst_20) & np.isfinite(alb) & np.isfinite(ndvi_20)
          & np.isfinite(mndwi_20) & np.isfinite(wc_20))
mask_loose  = finite & (to20(ix_b["ndbi"]) > 0)                    # the broken one
mask_strict = finite & bu20 & (wc_20 == WC_BUILTUP) & (mndwi_20 <= 0) & (ndvi_20 <= 0.20)
print(f"loose mask (NDBI>0 only) : {mask_loose.sum():,} px")
print(f"strict mask (RF AND WorldCover, no water/veg): {mask_strict.sum():,} px")

def fit(mask, label, K=5, block_m=1000, n_boot=300):
    y = lst_20[mask]; X = np.column_stack([alb[mask], ndvi_20[mask]])
    rr, cc = np.where(mask); bpx = int(block_m/RES_S2)
    bid = (rr//bpx)*10_000 + (cc//bpx); blocks = np.unique(bid)
    sh = blocks.copy(); np.random.shuffle(sh)
    fmap = {b:i%K for i,b in enumerate(sh)}
    fold = np.array([fmap[v] for v in bid])
    oof = np.full(len(y), np.nan)
    for k in range(K):
        a, b = fold!=k, fold==k
        if b.sum()<30 or a.sum()<100: continue
        oof[b] = LinearRegression().fit(X[a], y[a]).predict(X[b])
    okm = np.isfinite(oof)
    final = LinearRegression().fit(X, y)
    by_block = {b: np.where(bid==b)[0] for b in blocks}
    boot = []
    for _ in range(n_boot):
        pick = np.random.choice(blocks, len(blocks), True)
        ii = np.concatenate([by_block[b] for b in pick])
        boot.append(LinearRegression().fit(X[ii], y[ii]).coef_[0])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    res = dict(label=label, slope=float(final.coef_[0]),
               ci95=[float(lo), float(hi)], intercept=float(final.intercept_),
               ndvi_coef=float(final.coef_[1]), r2_in=float(final.score(X,y)),
               cv_r2=float(r2_score(y[okm], oof[okm])),
               cv_rmse=float(np.sqrt(mean_squared_error(y[okm], oof[okm]))),
               n=int(mask.sum()), n_blocks=int(len(blocks)))
    print(f"\n--- {label} ---")
    print(f"  n={res['n']:,} in {res['n_blocks']} blocks")
    print(f"  slope {res['slope']:+.2f} C per unit albedo   95% CI "
          f"[{lo:.2f}, {hi:.2f}]")
    print(f"  spatial-CV R2 {res['cv_r2']:.3f}   RMSE {res['cv_rmse']:.2f} C")
    return res, (y, X, oof, okm, np.array(boot))

res_loose, _ = fit(mask_loose, "LOOSE mask (NDBI>0) — the contaminated one", n_boot=50)
res_strict, (y, X, oof, okm, boot) = fit(mask_strict, "STRICT mask — used for the product")
SENS = res_strict["slope"]; ci_lo, ci_hi = res_strict["ci95"]

print("\n" + "="*72)
if SENS < 0 and ci_hi < 0:
    print(f"PHYSICS CHECK PASS: {SENS:.2f} C per unit albedo, CI entirely negative.")
    print(f"  +0.10 albedo => {abs(SENS)*0.10:.2f} C cooler "
          f"[{abs(ci_hi)*0.10:.2f}, {abs(ci_lo)*0.10:.2f}]")
    print("  Low R2 with a tight CI: albedo is one lever among many. We estimate its")
    print("  MARGINAL EFFECT, which is well constrained, not absolute temperature.")
else:
    print(f"PHYSICS CHECK FAIL: slope {SENS:+.2f}. Do not ship a cooling estimate.")
print("="*72)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, m, r, t in [(axes[0], mask_loose, res_loose, "Before: water + sabkha included"),
                    (axes[1], mask_strict, res_strict, "After: built-up only")]:
    yy, xx = lst_20[m], alb[m]
    i = np.random.choice(len(yy), min(20000, len(yy)), replace=False)
    ax.scatter(xx[i], yy[i], s=1.5, alpha=0.12, c="#1f3a5f")
    xs = np.linspace(np.percentile(xx,1), np.percentile(xx,99), 50)
    ax.plot(xs, r["intercept"] + r["slope"]*xs + r["ndvi_coef"]*np.median(ndvi_20[m]),
            c="#d32f2f", lw=2.2)
    ax.set_xlabel("Broadband albedo"); ax.set_ylabel("Summer LST (C)")
    ax.set_title(f"{t}\nslope {r['slope']:+.1f} C per unit albedo"); ax.grid(alpha=0.3)
axes[2].hist(boot, bins=40, color="#1f3a5f", alpha=0.85)
axes[2].axvline(SENS, color="#d32f2f", lw=2, label=f"estimate {SENS:.2f}")
axes[2].axvline(ci_lo, color="#d32f2f", ls="--", lw=1.2)
axes[2].axvline(ci_hi, color="#d32f2f", ls="--", lw=1.2, label="95% CI")
axes[2].axvline(0, color="k", lw=1)
axes[2].set_xlabel("Albedo coefficient (C per unit albedo)")
axes[2].set_ylabel("Block bootstrap resamples"); axes[2].legend()
axes[2].set_title("Spatial block bootstrap")
plt.tight_layout(); plt.savefig(f"{RESULTS}/04_albedo_model.png", bbox_inches="tight")
plt.show()

## 8 · From pixels to a worklist — ranking individual rooftops

```
ΔT      = |sensitivity| × max(0, 0.60 − current_albedo)      with CI bounds
benefit = ΔT × roof_area_m²                                   [°C·m²]
ΔQ      = (0.60 − current_albedo) × S × area                  [W avoided]
```

Area is in the formula deliberately: one large warehouse roof is far cheaper to treat
per square metre than fifty small ones.

In [ ]:
OVERPASS = ["https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter",
            "https://overpass-api.de/api/interpreter",
            "https://overpass.osm.jp/api/interpreter"]

def osm_buildings(bbox, attempts=2):
    w,s,e,n = bbox
    q = (f'[out:json][timeout:180];(way["building"]({s},{w},{n},{e});'
         f'relation["building"]({s},{w},{n},{e}););out geom;')
    data = None
    for att in range(attempts):
        for url in OVERPASS:
            host = url.split("/")[2]
            for verb in ("POST","GET"):
                try:
                    import requests
                    r = (requests.post(url, data={"data":q}, timeout=300) if verb=="POST"
                         else requests.get(url, params={"data":q}, timeout=300))
                    if r.status_code == 200:
                        data = r.json(); print(f"   {host} [{verb}] OK"); break
                    print(f"   {host} [{verb}] HTTP {r.status_code}")
                except Exception as ex:
                    print(f"   {host} [{verb}] {type(ex).__name__}")
            if data: break
        if data: break
        if att < attempts-1:
            print("   all mirrors busy, waiting 30s..."); time.sleep(30)
    if not data: raise RuntimeError("All Overpass mirrors failed — retry in a few minutes.")
    polys, props = [], []
    for el in data.get("elements", []):
        g = el.get("geometry")
        if not g or len(g) < 4: continue
        try:
            p = Polygon([(d["lon"], d["lat"]) for d in g])
            if p.is_valid and p.area > 0:
                polys.append(p); props.append({"osm_id": el.get("id")})
        except Exception: continue
    return gpd.GeoDataFrame(props, geometry=polys, crs="EPSG:4326")

print("Querying OpenStreetMap (1-3 minutes)...")
roofs = osm_buildings(FOCUS_BBOX).to_crs(epsg=EPSG)
roofs["area_m2"] = roofs.geometry.area
roofs = roofs[roofs.area_m2 >= 400].reset_index(drop=True)
print(f"{len(roofs):,} roofs >= 400 m2, {roofs.area_m2.sum()/1e6:.2f} km2 of roof")

In [ ]:
affine = albedo.rio.transform()
def zmean(arr):
    st = zonal_stats(roofs, np.asarray(arr, dtype="float32"), affine=affine,
                     stats=["mean"], nodata=np.nan)
    return np.array([s["mean"] if s and s["mean"] is not None else np.nan for s in st])

print("Zonal statistics per roof...")
roofs["albedo"] = zmean(alb)
roofs["lst_c"]  = zmean(lst_20)
roofs = roofs.dropna(subset=["albedo","lst_c"]).reset_index(drop=True)

s = abs(SENS)
roofs["albedo_gap"]    = (TARGET_ALBEDO - roofs.albedo).clip(lower=0)
roofs["delta_T_est_c"] = s * roofs.albedo_gap
roofs["delta_T_lo_c"]  = abs(ci_hi) * roofs.albedo_gap
roofs["delta_T_hi_c"]  = abs(ci_lo) * roofs.albedo_gap
roofs["cooling_c_m2"]  = roofs.delta_T_est_c * roofs.area_m2
roofs["lst_anomaly_c"] = roofs.lst_c - roofs.lst_c.median()
roofs["dQ_mean_kW"]    = roofs.albedo_gap * S_SUMMER_MEAN * roofs.area_m2 / 1000
roofs["dQ_peak_kW"]    = roofs.albedo_gap * S_NOON_PEAK   * roofs.area_m2 / 1000
roofs = roofs.sort_values("cooling_c_m2", ascending=False).reset_index(drop=True)
roofs["rank"] = np.arange(1, len(roofs)+1)
roofs["rank_benefit"]   = roofs.cooling_c_m2.rank(ascending=False).astype(int)
roofs["rank_hottest"]   = roofs.lst_c.rank(ascending=False).astype(int)
roofs["rank_intensity"] = roofs.delta_T_est_c.rank(ascending=False).astype(int)

cols = ["rank","osm_id","area_m2","albedo","lst_c","lst_anomaly_c","albedo_gap",
        "delta_T_est_c","delta_T_lo_c","delta_T_hi_c","cooling_c_m2",
        "dQ_mean_kW","dQ_peak_kW","rank_benefit","rank_hottest","rank_intensity"]
roofs[cols].round(3).to_csv(f"{RESULTS}/roof_priority_ranked.csv", index=False)
roofs.head(500)[cols+["geometry"]].to_file(f"{RESULTS}/top500_roofs.geojson", driver="GeoJSON")

top = roofs.head(100)
conc = (top.cooling_c_m2.sum()/roofs.cooling_c_m2.sum())/(100/len(roofs))
print(f"\n{len(roofs):,} roofs ranked")
print(f"Top 100 ({100/len(roofs)*100:.1f}% of stock): "
      f"{top.cooling_c_m2.sum()/roofs.cooling_c_m2.sum()*100:.1f}% of benefit, "
      f"{conc:.1f}x concentration")
display(roofs[cols[:11]].head(15).round(3))

print("\n" + "="*86)
print("RANKING MODES — the same data answers three different planner questions")
print("="*86)
for label, col in [("Total benefit (area x cooling)","rank_benefit"),
                   ("Hottest roofs first","rank_hottest"),
                   ("Biggest per-roof temperature drop","rank_intensity")]:
    sel = roofs.nsmallest(100, col)
    print(f"{label:36s} top100: {sel.area_m2.sum()/1e4:7.1f} ha | "
          f"albedo {sel.albedo.mean():.3f} | LST {sel.lst_c.mean():.1f} C | "
          f"{sel.dQ_mean_kW.sum()/1000:5.1f} MW")
overlap = len(set(roofs.nsmallest(100,"rank_benefit").osm_id) &
              set(roofs.nsmallest(100,"rank_hottest").osm_id))
print(f"\nOverlap between 'total benefit' and 'hottest' top-100: {overlap} roofs")
print("Low overlap means the ranking choice is a policy decision, not a technical one.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
roofs.plot(ax=axes[0], column="lst_c", cmap="inferno", legend=True,
           legend_kwds={"label":"Mean roof LST (C)","shrink":0.7})
axes[0].set_title("Every roof, by surface temperature"); axes[0].set_axis_off()
roofs.plot(ax=axes[1], color="#e8e8e8"); top.plot(ax=axes[1], color="#d32f2f")
axes[1].set_title("Top 100 priority roofs (red)"); axes[1].set_axis_off()
axes[2].scatter(roofs.albedo, roofs.lst_c, s=np.clip(roofs.area_m2/900,2,60),
                alpha=0.35, c="#1f3a5f", edgecolors="none")
axes[2].set_xlabel("Roof albedo"); axes[2].set_ylabel("Roof LST (C)")
axes[2].set_title("Roofs: albedo vs temperature\n(marker size = roof area)")
axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS}/05_roof_priority.png", bbox_inches="tight")
plt.show()

## 9 · Physical impact and export

Raising a roof's albedo by Δα reduces absorbed shortwave by `ΔQ = Δα · S · A` watts.
Megawatts avoided is a quantity an energy planner and a physicist both recognise, and
it ties the product back to the surface energy balance.

This is **avoided absorption, not an air-conditioning saving.** The two are related but
not equal and we make no energy-cost claim.

In [ ]:
tot_mean_MW = roofs.dQ_mean_kW.sum()/1000
print("="*78)
print("PHYSICAL IMPACT — absorbed solar power avoided at albedo 0.60")
print("="*78)
print(f"  all {len(roofs):,} roofs : {tot_mean_MW:8.1f} MW summer-mean | "
      f"{roofs.dQ_peak_kW.sum()/1000:8.1f} MW noon peak")
print(f"  top 100 roofs      : {top.dQ_mean_kW.sum()/1000:8.1f} MW summer-mean | "
      f"{top.dQ_peak_kW.sum()/1000:8.1f} MW noon peak")
print(f"  roof area treated  : {top.area_m2.sum()/1e4:,.0f} ha of "
      f"{roofs.area_m2.sum()/1e4:,.0f} ha")
print(f"\n  Assumes S = {S_SUMMER_MEAN:.0f} W/m2 [TEAM: verify and cite]")
print("="*78)

summary = {
 "aoi": AOI_NAME, "aoi_bbox": AOI_BBOX, "focus_bbox": FOCUS_BBOX,
 "epochs": {"before": EPOCH_A_NAME, "after": EPOCH_B_NAME},
 "scenes_used": {"landsat_before": len(items_a), "landsat_after": len(items_b),
                 "sentinel2": len(items_s2)},
 "mean_lst_before_c": round(float(np.nanmean(lst_a.values)),2),
 "mean_lst_after_c":  round(float(np.nanmean(lst_b.values)),2),
 "desert_to_builtup_km2": EXPANSION_KM2,
 "desert_to_builtup_hectares": round(EXPANSION_KM2*100),
 "static_uhi_c": round(uhi_static,2),
 "conversion_extra_warming_c": HEADLINE,
 "conversion_verdict": sig_rows[0]["verdict"],
 "pre_construction_temperature_gap_c": round(dT_pre,2),
 "significance_tests": sig_rows,
 "albedo_sensitivity_c_per_unit": round(SENS,2),
 "albedo_sensitivity_ci95": [round(ci_lo,2), round(ci_hi,2)],
 "cooling_per_0.1_albedo_c": round(abs(SENS)*0.1,2),
 "mask_sensitivity_check": {"loose_slope": round(res_loose["slope"],2),
   "strict_slope": round(SENS,2),
   "note": ("An NDBI>0 mask admits channel water (dark, cold) and bright sabkha "
            "(bright, hot), inverting the slope. Requiring agreement with an "
            "independent land-cover product recovers the expected negative slope.")},
 "validation_builtup": validation_builtup,
 "validation_model": {k:(round(v,4) if isinstance(v,float) else v)
                      for k,v in res_strict.items()},
 "roofs_analysed": int(len(roofs)),
 "roof_area_total_km2": round(float(roofs.area_m2.sum()/1e6),3),
 "top100_benefit_share_pct": round(float(top.cooling_c_m2.sum()/roofs.cooling_c_m2.sum()*100),1),
 "top100_concentration_x": round(float(conc),1),
 "physical_impact": {"assumed_irradiance_W_m2": S_SUMMER_MEAN,
   "avoided_absorption_all_MW": round(float(tot_mean_MW),1),
   "avoided_absorption_top100_MW": round(float(top.dQ_mean_kW.sum()/1000),1),
   "caveat": "Avoided shortwave absorption, NOT an air-conditioning energy saving."},
 "ranking_modes": {"benefit_vs_hottest_top100_overlap": int(overlap)},
}
with open(f"{RESULTS}/summary.json","w") as f: json.dump(summary, f, indent=2)

# sample input + provenance
try:
    bnds = gpd.GeoSeries([box(*FOCUS_BBOX)], crs=4326).to_crs(EPSG).total_bounds
    albedo.rio.clip_box(*bnds).rio.to_raster(f"{SAMPLE}/albedo_focus.tif", compress="deflate")
except Exception as e:
    print("sample clip failed:", e)
with open(f"{SAMPLE}/provenance.json","w") as f:
    json.dump({"generated_utc": pd.Timestamp.utcnow().isoformat(),
      "aoi_bbox": AOI_BBOX, "focus_bbox": FOCUS_BBOX, "epsg": EPSG,
      "landsat": {"collection":"landsat-c2-l2","windows":[EPOCH_A,EPOCH_B],
                  "max_cloud_pct":MAX_CLOUD,"max_scenes":MAX_SCENES_LS,
                  "scene_ids_before":[i.id for i in items_a],
                  "scene_ids_after":[i.id for i in items_b]},
      "sentinel2": {"collection":"sentinel-2-l2a","window":S2_WINDOW,
                    "bands":S2_BANDS,"scene_ids":[i.id for i in items_s2]},
      "worldcover": {"items":[i.id for i in wc_2021],"builtup_class":WC_BUILTUP},
      "osm": {"source":"Overpass API","bbox":FOCUS_BBOX,"min_roof_area_m2":400},
      "licences": {"Landsat":"Public domain (USGS)",
        "Sentinel-2":"Copernicus Sentinel data, free and open",
        "ESA WorldCover":"CC BY 4.0",
        "OpenStreetMap":"ODbL (c) OpenStreetMap contributors"}}, f, indent=2)

print(json.dumps(summary, indent=2)[:2500], "...")
print("\nFiles written:")
for d in (RESULTS, SAMPLE):
    for fn in sorted(os.listdir(d)):
        print(f"   {d}/{fn:38s} {os.path.getsize(os.path.join(d,fn))/1e3:9.1f} KB")

In [ ]:
# Zip EVERYTHING the run produced — results/ AND data/sample_input/ — and push it
# to your Downloads folder. The sample_input folder holds provenance.json (every
# scene ID and query parameter) and albedo_focus.tif, which the README cites as the
# worked example; zipping only results/ leaves them behind on a runtime that dies.
import shutil
stage = "/content/_bundle" if os.path.isdir("/content") else "./_bundle"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(RESULTS, f"{stage}/results")
if os.path.isdir(SAMPLE):
    shutil.copytree(SAMPLE, f"{stage}/data/sample_input")
base = "/content/main_results" if os.path.isdir("/content") else "./main_results"
shutil.make_archive(base, "zip", stage)
p = base + ".zip"
print(f"zipped {p} ({os.path.getsize(p)/1e3:.0f} KB)")
for root, _, fs in os.walk(stage):
    for f in sorted(fs):
        print("   ", os.path.relpath(os.path.join(root, f), stage))
try:
    from google.colab import files; files.download(p)
    print("download started")
except Exception:
    print(f"grab it from the Files pane: {p}")

---

## Headline numbers

All of it is in `results/summary.json`. The limitations are set out in README §9.6 —
read them before quoting any figure. In particular: land surface temperature is not air
temperature, the albedo sensitivity is correlational rather than causal, and validation
is against an independent model rather than ground truth.
